# LeetCode 4108 — Minimum Cost to Make Red Nodes Connected (Medium)

Bài toán LeetCode:

```python
class Solution:
    def minCost(self, n: int, colors: str, edges: List[List[int]]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4108 (Weekly Contest 527 — câu 3, độ khó thực tế khá cao, rating khoảng 2015).

Đây là bài **quy hoạch động trên cây (tree DP)**. Notebook sẽ đi từng bước từ con số 0:

1. Ôn lại cây, gốc, cây con, kích thước cây con, duyệt cây không đệ quy.
2. **Biến đổi chi phí**: từ "lật màu" sang "chọn một tập đỉnh liên thông có tổng trọng số nhỏ nhất".
3. Quan sát then chốt: **mỗi tập liên thông trong cây có gốc có đúng một đỉnh cao nhất**.
4. Từ vét cạn `O(2^n)` → `O(n²)` → `O(n)`.

---

## Sau notebook này, bạn sẽ nắm được

1. Biểu diễn cây bằng **danh sách kề**, chọn gốc, tìm cha/con, tính **kích thước cây con**.
2. Duyệt cây bằng **stack** thay vì đệ quy (tránh `RecursionError` khi cây sâu `10^5`).
3. Mẹo **đổi mốc chi phí**: bắt đầu từ phương án "lật hết đỏ thành xanh", rồi tính tiền lời/lỗ khi giữ lại một tập đỏ.
4. Kỹ thuật DP **"tập liên thông tốt nhất có đỉnh cao nhất là `u`"**: `f(u) = w(u) + Σ min(0, f(con))`.
5. Chứng minh, độ phức tạp, kiểm thử chéo với vét cạn.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho một **cây** gồm `n` đỉnh đánh số `0` đến `n - 1`, **gốc là đỉnh `0`**. Cây được cho bởi `edges` gồm `n - 1` cạnh **vô hướng** `[u, v]`.

Chuỗi `colors` dài `n`: `colors[i]` là `'R'` (đỏ) hoặc `'B'` (xanh) — màu của đỉnh `i`.

**Thao tác:** chọn một đỉnh `u` bất kỳ và **lật màu** của nó (`R ↔ B`). Chi phí của thao tác bằng **kích thước cây con của `u`** (số đỉnh trong cây con gốc `u`, tính cả `u`, khi cây có gốc `0`).

**Mục tiêu:** sau các thao tác, các đỉnh đỏ phải tạo thành **đúng một thành phần liên thông** (đi được từ đỉnh đỏ này tới đỉnh đỏ kia mà chỉ qua đỉnh đỏ). Các đỉnh xanh không có yêu cầu gì. **Không còn đỉnh đỏ nào cũng được coi là hợp lệ.**

Trả về **tổng chi phí nhỏ nhất**.

---

## Ví dụ 1

```python
n = 3, colors = "RBR", edges = [[0, 1], [1, 2]]   ->   1
```

```text
0(R) — 1(B) — 2(R)
```

Đỉnh đỏ `0` và `2` bị đỉnh xanh `1` chặn. Lật đỉnh `2` (cây con chỉ có chính nó, kích thước 1) → chỉ còn đỉnh đỏ `0`. Chi phí `1`.

## Ví dụ 2

```python
n = 3, colors = "RBR", edges = [[0, 1], [0, 2]]   ->   0
```

```text
     0(R)
    /    \
  1(B)   2(R)
```

Đỉnh đỏ `0` và `2` nối trực tiếp với nhau → đã liên thông, chi phí `0`.

## Ví dụ 3

```python
n = 4, colors = "BRRR", edges = [[0, 1], [0, 2], [0, 3]]   ->   2
```

Ba lá đỏ `1, 2, 3` rời nhau (gốc `0` xanh). Giữ một lá, lật hai lá còn lại, mỗi lá tốn `1` → tổng `2`.

(Lật gốc `0` thành đỏ sẽ nối được cả ba, nhưng tốn bằng kích thước cây con của gốc là `4` — đắt hơn.)

## Ví dụ 4

```python
n = 3, colors = "BBB", edges = [[0, 1], [1, 2]]   ->   0
```

Không có đỉnh đỏ nào — hợp lệ, chi phí `0`.

---

## Ràng buộc

```text
1 <= n <= 10^5
colors.length == n, colors[i] là 'R' hoặc 'B'
edges.length == n - 1, edges[i] = [u_i, v_i], 0 <= u_i, v_i <= n - 1
edges luôn tạo thành một cây hợp lệ
```

# 2. Kiến thức nền tảng về cây

## 2.1. Cây, gốc, cha, con, cây con

- **Cây** với `n` đỉnh có đúng `n - 1` cạnh, liên thông và không có chu trình. Giữa hai đỉnh bất kỳ có **đúng một đường đi**.
- Chọn đỉnh `0` làm **gốc**, mỗi đỉnh khác có một **cha** (đỉnh kề nằm trên đường đi về gốc). Các đỉnh kề còn lại là **con**.
- **Cây con của `u`** gồm `u` và mọi đỉnh nằm "bên dưới" `u`. **Kích thước** cây con `size[u]` là số đỉnh trong đó.

Ví dụ xuyên suốt của notebook (7 đỉnh):

```text
              0(B)              size[0] = 7
            /      \
        1(R)        2(B)        size[1] = 3, size[2] = 3
       /    \      /    \
    3(R)   4(R)  5(B)   6(R)    các lá: size = 1
```

`edges = [[0,1],[0,2],[1,3],[1,4],[2,5],[2,6]]`, `colors = "BRBRRBR"`.

Lật đỉnh `1` tốn `3`, lật gốc `0` tốn `7`, lật một lá tốn `1`. Đỉnh càng gần gốc lật càng đắt.

## 2.2. Danh sách kề (adjacency list)

Từ danh sách cạnh, ta lưu cho mỗi đỉnh danh sách các đỉnh kề. Cạnh **vô hướng** nên thêm cả hai chiều.

In [1]:
n = 7
edges = [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]]
colors = "BRBRRBR"

adj = [[] for _ in range(n)]
for u, v in edges:
    adj[u].append(v)
    adj[v].append(u)

for u in range(n):
    print(f"đỉnh {u} ({colors[u]}): kề với {adj[u]}")

đỉnh 0 (B): kề với [1, 2]
đỉnh 1 (R): kề với [0, 3, 4]
đỉnh 2 (B): kề với [0, 5, 6]
đỉnh 3 (R): kề với [1]
đỉnh 4 (R): kề với [1]
đỉnh 5 (B): kề với [2]
đỉnh 6 (R): kề với [2]


## 2.3. Duyệt cây bằng stack (không đệ quy)

Ta cần biết **cha** của mỗi đỉnh và một **thứ tự duyệt** trong đó cha luôn đứng trước con.

Vì sao không dùng đệ quy? Cây có thể là một đường thẳng dài `10^5` đỉnh. Đệ quy sâu `10^5` tầng vượt giới hạn mặc định của Python (khoảng 1000 tầng) → `RecursionError`. Dùng **stack tự quản lý** thì không bị giới hạn này.

Thuật toán:

1. Đẩy gốc `0` vào stack, đánh dấu đã thăm.
2. Lặp: lấy một đỉnh `u` ra, ghi vào `order`; với mỗi đỉnh kề `v` chưa thăm → `parent[v] = u`, đánh dấu, đẩy vào stack.

Mỗi đỉnh được ghi vào `order` **sau** cha của nó (vì nó chỉ được đẩy vào stack khi cha đã được lấy ra).

In [2]:
def root_tree(n: int, edges):
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    parent = [-1] * n
    order = []
    visited = [False] * n
    visited[0] = True
    stack = [0]
    while stack:
        u = stack.pop()
        order.append(u)
        for v in adj[u]:
            if not visited[v]:
                visited[v] = True
                parent[v] = u
                stack.append(v)
    return parent, order


parent, order = root_tree(n, edges)
print("parent =", parent)
print("order  =", order, "(cha luôn đứng trước con)")

parent = [-1, 0, 0, 1, 1, 2, 2]
order  = [0, 2, 6, 5, 1, 4, 3] (cha luôn đứng trước con)


## 2.4. Tính kích thước cây con: duyệt `order` theo chiều ngược

Duyệt `order` **từ cuối lên đầu** thì mọi đỉnh con được xử lý **trước** cha. Khi xử lý `u`, `size[u]` đã đủ (gồm chính nó và các con đã cộng vào), ta cộng tiếp `size[u]` vào `size[parent[u]]`.

In [3]:
size = [1] * n
for u in reversed(order):
    if parent[u] != -1:
        size[parent[u]] += size[u]

print("size =", size)

size = [7, 3, 3, 1, 1, 1, 1]


## 2.5. Kiểm tra một tập đỉnh có liên thông không

Dùng cho cách vét cạn: bắt đầu từ một đỉnh trong tập, chỉ đi qua các đỉnh **trong tập**, xem có thăm hết tập không.

In [4]:
def is_connected_set(nodes: set, adj) -> bool:
    if not nodes:
        return True                          # tập rỗng: coi là hợp lệ
    start = next(iter(nodes))
    seen = {start}
    stack = [start]
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v in nodes and v not in seen:
                seen.add(v)
                stack.append(v)
    return len(seen) == len(nodes)


print(is_connected_set({1, 3, 4}, adj))   # True: 3 - 1 - 4
print(is_connected_set({1, 6}, adj))      # False: cần đi qua 0 và 2
print(is_connected_set(set(), adj))       # True

True
False
True


# 3. Quan sát đầu tiên: chỉ "trạng thái cuối" mới quan trọng

- Lật một đỉnh **hai lần** thì màu như cũ nhưng tốn gấp đôi → không bao giờ có lợi. Vậy mỗi đỉnh được lật **0 hoặc 1 lần**.
- Thứ tự lật không ảnh hưởng gì (chi phí mỗi lần lật chỉ phụ thuộc kích thước cây con, vốn cố định).

Do đó, một phương án được xác định hoàn toàn bởi **tập đỉnh đỏ cuối cùng** `S`:

- đỉnh **xanh** nằm trong `S` → phải lật (B → R), tốn `size[u]`;
- đỉnh **đỏ** nằm ngoài `S` → phải lật (R → B), tốn `size[u]`;
- các đỉnh khác giữ nguyên.

```text
cost(S) = Σ size[u] với u ∈ S mà u xanh   +   Σ size[u] với u ∉ S mà u đỏ
```

Bài toán trở thành: **chọn tập `S` liên thông (hoặc rỗng) sao cho `cost(S)` nhỏ nhất.**

---

# 4. Cách 1 — Vét cạn mọi tập `S`

Thử cả `2^n` tập con, giữ những tập liên thông, tính `cost(S)`.

In [5]:
from typing import List


def min_cost_bruteforce(n: int, colors: str, edges: List[List[int]]) -> int:
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    parent, order = root_tree(n, edges)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]

    best = float("inf")
    for mask in range(1 << n):                          # mỗi mask là một tập S
        S = {u for u in range(n) if mask >> u & 1}
        if not is_connected_set(S, adj):
            continue
        cost = 0
        for u in range(n):
            in_S = u in S
            is_red = colors[u] == 'R'
            if in_S != is_red:                          # màu cuối khác màu đầu -> phải lật
                cost += size[u]
        best = min(best, cost)
    return best


print(min_cost_bruteforce(3, "RBR", [[0, 1], [1, 2]]))            # 1
print(min_cost_bruteforce(3, "RBR", [[0, 1], [0, 2]]))            # 0
print(min_cost_bruteforce(4, "BRRR", [[0, 1], [0, 2], [0, 3]]))   # 2
print(min_cost_bruteforce(3, "BBB", [[0, 1], [1, 2]]))            # 0
print(min_cost_bruteforce(7, "BRBRRBR", edges))                   # ví dụ xuyên suốt

1
0
2
0
1


`2^n` tập con với `n = 10^5` là con số không tưởng (`2^20` đã là một triệu). Ta cần hiểu cấu trúc của bài toán sâu hơn.

---

# 5. Bước đột phá 1: đổi mốc chi phí

Hãy bắt đầu từ một phương án **luôn hợp lệ**: **lật hết mọi đỉnh đỏ thành xanh** (`S` rỗng). Chi phí:

```text
total_red = Σ size[u] với mọi u đỏ
```

Bây giờ, so với phương án mốc này, việc đưa một đỉnh `u` vào `S` làm chi phí thay đổi bao nhiêu?

- `u` **đỏ**: ở mốc ta đã trả `size[u]` để lật nó. Nếu `u ∈ S` thì **không lật nữa** → **tiết kiệm** `size[u]`.
- `u` **xanh**: ở mốc không tốn gì. Nếu `u ∈ S` thì phải lật thành đỏ → **tốn thêm** `size[u]`.

Đặt **trọng số** cho mỗi đỉnh:

```text
w[u] = +size[u]   nếu u xanh   (đưa vào S thì tốn thêm)
w[u] = -size[u]   nếu u đỏ     (đưa vào S thì tiết kiệm)
```

Khi đó:

```text
cost(S) = total_red + Σ_{u ∈ S} w[u]
```

> Bài toán trở thành: **chọn tập liên thông `S` (có thể rỗng) có tổng trọng số `Σ w[u]` nhỏ nhất.** Vì `S` rỗng cho tổng `0`, đáp án luôn `<= total_red`.

In [6]:
w = [size[u] if colors[u] == 'B' else -size[u] for u in range(n)]
total_red = sum(size[u] for u in range(n) if colors[u] == 'R')

print("size      =", size)
print("colors    =", list(colors))
print("w         =", w)
print("total_red =", total_red)

# kiểm tra công thức cost(S) = total_red + sum(w) với vài tập S
for S in [set(), {1, 3, 4}, {0, 1, 2, 3, 4, 6}, {6}]:
    print(f"S = {str(sorted(S)):20} -> total_red + Σw = {total_red + sum(w[u] for u in S)}")

size      = [7, 3, 3, 1, 1, 1, 1]
colors    = ['B', 'R', 'B', 'R', 'R', 'B', 'R']
w         = [7, -3, 3, -1, -1, 1, -1]
total_red = 6
S = []                   -> total_red + Σw = 6
S = [1, 3, 4]            -> total_red + Σw = 1
S = [0, 1, 2, 3, 4, 6]   -> total_red + Σw = 10
S = [6]                  -> total_red + Σw = 5


Đọc kết quả với ví dụ xuyên suốt:

- `S = {}`: lật hết 4 đỉnh đỏ → `3 + 1 + 1 + 1 = 6`.
- `S = {1, 3, 4}`: giữ nhánh đỏ bên trái, chỉ lật lá đỏ `6` → `1`.
- `S = {0,1,2,3,4,6}`: phải lật gốc `0` (7) và đỉnh `2` (3) thành đỏ để nối tất cả → `10`.

---

# 6. Bước đột phá 2: mỗi tập liên thông có đúng một "đỉnh cao nhất"

Trong cây có gốc, mỗi đỉnh có một **độ sâu** (khoảng cách tới gốc).

> **Tính chất:** mỗi tập liên thông khác rỗng `S` có **đúng một** đỉnh có độ sâu nhỏ nhất. Gọi đó là **đỉnh cao nhất (top)** của `S`.

Vì sao? Giả sử có hai đỉnh `x ≠ y` trong `S` cùng độ sâu nhỏ nhất. Đường đi duy nhất giữa `x` và `y` phải đi **lên** qua cha của chúng — một đỉnh **nông hơn**. Vì `S` liên thông, đường đi đó nằm trọn trong `S` → `S` chứa một đỉnh nông hơn `x` — mâu thuẫn.

Hệ quả: nếu top của `S` là `t`, thì `S` nằm trọn trong cây con của `t`, chứa `t`, và liên thông. Ta có thể **phân loại mọi tập `S` theo đỉnh top** của nó.

## Tập tốt nhất có top là `u`

Gọi `f(u)` = tổng trọng số **nhỏ nhất** của một tập liên thông có top là `u`.

Tập như vậy chắc chắn chứa `u`. Với mỗi con `c` của `u`, chỉ có hai lựa chọn:

- `S` **không đi xuống** cây con của `c` → đóng góp `0`;
- `S` **có đi xuống** cây con của `c` → để liên thông với `u`, nó phải chứa `c`; phần nằm trong cây con của `c` là một tập liên thông có top `c`, tốt nhất là `f(c)`.

Các con độc lập với nhau, nên:

```text
f(u) = w[u] + Σ_{c là con của u} min(0, f(c))
```

Câu chữ dễ nhớ: **"nhánh con nào có lời (âm) thì lấy, nhánh nào lỗ (dương) thì bỏ".**

Đáp án:

```text
answer = total_red + min(0, min_u f(u))
```

(`0` tương ứng với `S` rỗng.)

# 7. Cách 2 — Thử từng đỉnh làm top: O(n²)

Cài đặt trực tiếp ý tưởng trên: với **mỗi** đỉnh `t`, duyệt riêng cây con của `t` để tính `f` theo công thức, lấy `f(t)`.

In [7]:
def min_cost_quadratic(n: int, colors: str, edges: List[List[int]]) -> int:
    parent, order = root_tree(n, edges)
    children = [[] for _ in range(n)]
    for u in range(n):
        if parent[u] != -1:
            children[parent[u]].append(u)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]
    w = [size[u] if colors[u] == 'B' else -size[u] for u in range(n)]
    total_red = sum(size[u] for u in range(n) if colors[u] == 'R')

    best_sum = 0                                   # S rỗng
    for t in range(n):                             # thử t làm top
        sub = []                                   # các đỉnh trong cây con của t
        stack = [t]
        while stack:
            u = stack.pop()
            sub.append(u)
            stack.extend(children[u])
        f = {}
        for u in reversed(sub):                    # con trước cha
            f[u] = w[u] + sum(min(0, f[c]) for c in children[u])
        best_sum = min(best_sum, f[t])
    return total_red + best_sum


print(min_cost_quadratic(3, "RBR", [[0, 1], [1, 2]]))            # 1
print(min_cost_quadratic(3, "RBR", [[0, 1], [0, 2]]))            # 0
print(min_cost_quadratic(4, "BRRR", [[0, 1], [0, 2], [0, 3]]))   # 2
print(min_cost_quadratic(3, "BBB", [[0, 1], [1, 2]]))            # 0
print(min_cost_quadratic(7, "BRBRRBR", edges))                   # 1

1
0
2
0
1


## Lãng phí ở đâu?

Với cây là một đường thẳng `0 — 1 — 2 — ... — (n-1)`, cây con của đỉnh `t` có `n - t` đỉnh, nên tổng công việc là `n + (n-1) + ... + 1 ≈ n²/2`. Với `n = 10^5` là 5 tỷ bước.

Nhưng hãy nhìn lại công thức:

```text
f(u) = w[u] + Σ min(0, f(c))
```

**`f(u)` chỉ phụ thuộc vào cây con của `u`, không hề phụ thuộc vào `t`!** Khi thử `t = 0` ta đã tính `f` của **mọi** đỉnh rồi. Các lần thử `t` khác chỉ tính lại y hệt.

---

# 8. Cách 3 — Tree DP một lượt: O(n)

Tính `f(u)` cho mọi `u` **một lần duy nhất**, theo thứ tự con trước cha (`reversed(order)`), và lấy `min` trong lúc tính.

Cài đặt gọn: dùng mảng `f` khởi tạo `0`; khi xử lý `u`:

1. `f[u] += w[u]` (lúc này `f[u]` đã chứa tổng `min(0, f(con))` do các con cộng vào);
2. cập nhật `best = min(best, f[u])`;
3. cộng `min(0, f[u])` vào `f[parent[u]]`.

In [8]:
def min_cost_tree_dp(n: int, colors: str, edges: List[List[int]]) -> int:
    parent, order = root_tree(n, edges)

    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]

    total_red = 0
    f = [0] * n                      # f[u] tích lũy dần: Σ min(0, f(con)) rồi + w[u]
    best = 0                         # S rỗng
    for u in reversed(order):        # con trước cha
        if colors[u] == 'R':
            total_red += size[u]
            f[u] -= size[u]          # w[u] = -size[u]
        else:
            f[u] += size[u]          # w[u] = +size[u]
        best = min(best, f[u])
        if parent[u] != -1:
            f[parent[u]] += min(0, f[u])
    return total_red + best


print(min_cost_tree_dp(3, "RBR", [[0, 1], [1, 2]]))            # 1
print(min_cost_tree_dp(3, "RBR", [[0, 1], [0, 2]]))            # 0
print(min_cost_tree_dp(4, "BRRR", [[0, 1], [0, 2], [0, 3]]))   # 2
print(min_cost_tree_dp(3, "BBB", [[0, 1], [1, 2]]))            # 0
print(min_cost_tree_dp(7, "BRBRRBR", edges))                   # 1

1
0
2
0
1


## Mô phỏng với ví dụ xuyên suốt

```text
              0(B)
            /      \
        1(R)        2(B)
       /    \      /    \
    3(R)   4(R)  5(B)   6(R)
```

`size = [7, 3, 3, 1, 1, 1, 1]`, `w = [7, -3, 3, -1, -1, 1, -1]`, `total_red = 6`.

Thứ tự xử lý (con trước cha): `3, 4, 1, 5, 6, 2, 0`.

| Đỉnh u | w[u] | Đóng góp từ các con `min(0, f(c))` | f(u) | Ghi chú |
|---:|---:|---|---:|---|
| 3 | -1 | (lá) | -1 | |
| 4 | -1 | (lá) | -1 | |
| 1 | -3 | con 3: -1, con 4: -1 | **-5** | lấy cả hai nhánh đỏ |
| 5 | +1 | (lá) | 1 | lá xanh: lỗ |
| 6 | -1 | (lá) | -1 | |
| 2 | +3 | con 5: `min(0, 1) = 0` (bỏ), con 6: -1 | 2 | |
| 0 | +7 | con 1: -5, con 2: `min(0, 2) = 0` (bỏ) | 2 | |

`min f = -5` tại đỉnh `1` → tập tốt nhất là `S = {1, 3, 4}`.

Đáp án: `total_red + min(0, -5) = 6 - 5 = 1` — giữ nhánh đỏ bên trái, chỉ lật lá đỏ `6` (tốn `1`).

In [9]:
def trace_tree_dp(n: int, colors: str, edges: List[List[int]]) -> int:
    parent, order = root_tree(n, edges)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]
    total_red = sum(size[u] for u in range(n) if colors[u] == 'R')
    print(f"size = {size}, total_red = {total_red}")
    f = [0] * n
    best, best_node = 0, None
    for u in reversed(order):
        from_children = f[u]
        w_u = size[u] if colors[u] == 'B' else -size[u]
        f[u] += w_u
        if f[u] < best:
            best, best_node = f[u], u
        note = ""
        if parent[u] != -1:
            take = min(0, f[u])
            f[parent[u]] += take
            note = f"-> gửi lên cha {parent[u]}: {take}" + (" (bỏ nhánh)" if take == 0 else " (lấy nhánh)")
        print(f"u = {u} ({colors[u]}): w = {w_u:>3}, từ các con = {from_children:>3}, f = {f[u]:>3} {note}")
    print(f"min f = {best} (top = {best_node}) -> đáp án = {total_red} + ({best}) = {total_red + best}")
    return total_red + best


trace_tree_dp(7, "BRBRRBR", edges);

size = [7, 3, 3, 1, 1, 1, 1], total_red = 6
u = 3 (R): w =  -1, từ các con =   0, f =  -1 -> gửi lên cha 1: -1 (lấy nhánh)
u = 4 (R): w =  -1, từ các con =   0, f =  -1 -> gửi lên cha 1: -1 (lấy nhánh)
u = 1 (R): w =  -3, từ các con =  -2, f =  -5 -> gửi lên cha 0: -5 (lấy nhánh)
u = 5 (B): w =   1, từ các con =   0, f =   1 -> gửi lên cha 2: 0 (bỏ nhánh)
u = 6 (R): w =  -1, từ các con =   0, f =  -1 -> gửi lên cha 2: -1 (lấy nhánh)
u = 2 (B): w =   3, từ các con =  -1, f =   2 -> gửi lên cha 0: 0 (bỏ nhánh)
u = 0 (B): w =   7, từ các con =  -5, f =   2 
min f = -5 (top = 1) -> đáp án = 6 + (-5) = 1


# 9. Phiên bản đúng format LeetCode

In [10]:
from typing import List


class Solution:
    def minCost(self, n: int, colors: str, edges: List[List[int]]) -> int:
        adj = [[] for _ in range(n)]
        for u, v in edges:
            adj[u].append(v)
            adj[v].append(u)

        parent = [-1] * n
        order = []
        visited = [False] * n
        visited[0] = True
        stack = [0]
        while stack:
            u = stack.pop()
            order.append(u)
            for v in adj[u]:
                if not visited[v]:
                    visited[v] = True
                    parent[v] = u
                    stack.append(v)

        size = [1] * n
        for u in reversed(order):
            if parent[u] != -1:
                size[parent[u]] += size[u]

        total_red = 0
        f = [0] * n
        best = 0
        for u in reversed(order):
            if colors[u] == 'R':
                total_red += size[u]
                f[u] -= size[u]
            else:
                f[u] += size[u]
            best = min(best, f[u])
            if parent[u] != -1:
                f[parent[u]] += min(0, f[u])
        return total_red + best

In [11]:
sol = Solution()

tests = [
    (3, "RBR", [[0, 1], [1, 2]], 1, "ví dụ 1"),
    (3, "RBR", [[0, 1], [0, 2]], 0, "ví dụ 2"),
    (4, "BRRR", [[0, 1], [0, 2], [0, 3]], 2, "ví dụ 3"),
    (3, "BBB", [[0, 1], [1, 2]], 0, "ví dụ 4"),
    (1, "R", [], 0, "một đỉnh đỏ"),
    (1, "B", [], 0, "một đỉnh xanh"),
    (4, "RRRR", [[0, 1], [1, 2], [2, 3]], 0, "toàn đỏ, đã liên thông"),
    (3, "RBR", [[1, 0], [2, 1]], 1, "cạnh ghi ngược chiều"),
    (7, "BRBRRBR", [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]], 1, "ví dụ xuyên suốt"),
    (5, "RBRBR", [[0, 1], [1, 2], [2, 3], [3, 4]], 4, "đường thẳng đỏ-xanh xen kẽ"),
]

for n_, colors_, edges_, expected, name in tests:
    result = sol.minCost(n_, colors_, edges_)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:30} -> {result}")

PASS | ví dụ 1                        -> 1
PASS | ví dụ 2                        -> 0
PASS | ví dụ 3                        -> 2
PASS | ví dụ 4                        -> 0
PASS | một đỉnh đỏ                    -> 0
PASS | một đỉnh xanh                  -> 0
PASS | toàn đỏ, đã liên thông         -> 0
PASS | cạnh ghi ngược chiều           -> 1
PASS | ví dụ xuyên suốt               -> 1
PASS | đường thẳng đỏ-xanh xen kẽ     -> 4


## Kiểm tra bằng tay test cuối `RBRBR`

Cây là đường thẳng `0(R) — 1(B) — 2(R) — 3(B) — 4(R)`, kích thước cây con lần lượt `5, 4, 3, 2, 1`.

| Tập đỏ cuối `S` | Phải lật | Chi phí |
|---|---|---:|
| `{0}` | `2`, `4` (đỏ → xanh) | `3 + 1 = 4` |
| `{0, 1, 2}` | `1` (xanh → đỏ), `4` | `4 + 1 = 5` |
| `{2}` | `0`, `4` | `5 + 1 = 6` |
| `{2, 3, 4}` | `0`, `3` (xanh → đỏ) | `5 + 2 = 7` |
| `{4}` | `0`, `2` | `5 + 3 = 8` |
| `{}` | `0`, `2`, `4` | `9` |

Rẻ nhất là `4`: **giữ gốc** — đỉnh đỏ đắt nhất nếu phải lật — và lật hai đỉnh đỏ phía dưới. So sánh với vét cạn và trace của DP:

In [12]:
print("Vét cạn:", min_cost_bruteforce(5, "RBRBR", [[0, 1], [1, 2], [2, 3], [3, 4]]))
trace_tree_dp(5, "RBRBR", [[0, 1], [1, 2], [2, 3], [3, 4]]);

Vét cạn: 4
size = [5, 4, 3, 2, 1], total_red = 9
u = 4 (R): w =  -1, từ các con =   0, f =  -1 -> gửi lên cha 3: -1 (lấy nhánh)
u = 3 (B): w =   2, từ các con =  -1, f =   1 -> gửi lên cha 2: 0 (bỏ nhánh)
u = 2 (R): w =  -3, từ các con =   0, f =  -3 -> gửi lên cha 1: -3 (lấy nhánh)
u = 1 (B): w =   4, từ các con =  -3, f =   1 -> gửi lên cha 0: 0 (bỏ nhánh)
u = 0 (R): w =  -5, từ các con =   0, f =  -5 
min f = -5 (top = 0) -> đáp án = 9 + (-5) = 4


Trace cho thấy: ở gốc, nhánh con `1` có `f(1) = 1 > 0` (nối xuống phía dưới thì lỗ) nên bị cắt; tập tốt nhất là `S = {0}` với `f(0) = -5`, đáp án `9 - 5 = 4`. DP tự động so sánh mọi phương án như trong bảng trên mà không cần liệt kê — tính tay thì rất dễ bỏ sót.

---

# 10. Vì sao thuật toán đúng?

1. **Chỉ trạng thái cuối quan trọng** (mục 3): mỗi đỉnh lật tối đa một lần, chi phí = `Σ size` của các đỉnh đổi màu.
2. **Đổi mốc** (mục 5): `cost(S) = total_red + Σ_{u∈S} w[u]` với mọi tập `S`.
3. **Phân loại theo top** (mục 6): mọi tập liên thông khác rỗng có đúng một top `t`; tập rỗng có tổng `0`.
4. **Công thức `f`**: một tập liên thông có top `u` gồm `u` và, với mỗi con `c`, hoặc không gì trong cây con của `c`, hoặc một tập liên thông có top `c`. Các con độc lập → tổng nhỏ nhất là `w[u] + Σ min(0, f(c))`. (Chứng minh bằng quy nạp từ lá lên.)
5. **Thứ tự tính**: `reversed(order)` đảm bảo mọi con được tính xong trước cha.

Vậy `min(0, min_u f(u))` đúng là tổng trọng số nhỏ nhất trên mọi tập liên thông (kể cả rỗng), và đáp án là `total_red` cộng với nó.

---

## Trực giác

> Mặc định ta "hy sinh" mọi đỉnh đỏ. Rồi ta đi tìm **một cụm liên thông đáng giữ nhất**: cụm đó gom được nhiều đỉnh đỏ "đắt" (gần gốc, cây con to) mà phải trả ít tiền lật đỉnh xanh để nối chúng. Ở mỗi đỉnh, nhánh con nào mang lại lời thì kéo vào cụm, nhánh nào lỗ thì cắt bỏ.

# 11. Độ phức tạp

Gọi `n` là số đỉnh.

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ |
|---|---|---:|---:|
| 1. Vét cạn | thử mọi tập `S`, kiểm tra liên thông | `O(2^n · n)` | `O(n)` |
| 2. Thử từng top | DP lại trong cây con của mỗi `t` | `O(n²)` | `O(n)` |
| 3. Tree DP một lượt | tính `f(u)` cho mọi `u` một lần | `O(n)` | `O(n)` |

Cách 3 gồm: dựng danh sách kề `O(n)`, duyệt stack `O(n)`, tính `size` `O(n)`, tính `f` `O(n)`.

# 12. Đối chiếu ba cách trên cây ngẫu nhiên nhỏ

Sinh cây ngẫu nhiên bằng cách nối mỗi đỉnh `i >= 1` với một đỉnh ngẫu nhiên `< i`, và đảo chiều cạnh ngẫu nhiên để chắc chắn code không phụ thuộc chiều ghi cạnh.

In [13]:
import random

random.seed(4108)
for _ in range(1500):
    n_ = random.randint(1, 9)
    edges_ = []
    for i in range(1, n_):
        p = random.randrange(i)
        edges_.append([p, i] if random.random() < 0.5 else [i, p])
    colors_ = "".join(random.choice("RB") for _ in range(n_))

    expected = min_cost_bruteforce(n_, colors_, edges_)
    assert min_cost_quadratic(n_, colors_, edges_) == expected
    assert min_cost_tree_dp(n_, colors_, edges_) == expected
    assert sol.minCost(n_, colors_, edges_) == expected

print("1500 cây ngẫu nhiên: 4 hàm cho cùng kết quả!")

1500 cây ngẫu nhiên: 4 hàm cho cùng kết quả!


Và thử trên cây **rất sâu** (đường thẳng `10^5` đỉnh) cùng cây ngẫu nhiên `10^5` đỉnh để chắc chắn không có `RecursionError` và đủ nhanh:

In [14]:
import time

N = 10**5
chain_edges = [[i, i + 1] for i in range(N - 1)]
chain_colors = "".join(random.choice("RB") for _ in range(N))
t0 = time.perf_counter()
r1 = sol.minCost(N, chain_colors, chain_edges)
t1 = time.perf_counter() - t0

rand_edges = [[random.randrange(i), i] for i in range(1, N)]
rand_colors = "".join(random.choice("RB") for _ in range(N))
t0 = time.perf_counter()
r2 = sol.minCost(N, rand_colors, rand_edges)
t2 = time.perf_counter() - t0

print(f"đường thẳng 10^5 đỉnh: {r1:,} ({t1:.2f}s) | cây ngẫu nhiên 10^5 đỉnh: {r2:,} ({t2:.2f}s)")

đường thẳng 10^5 đỉnh: 2,472,517,517 (0.08s) | cây ngẫu nhiên 10^5 đỉnh: 442,634 (0.14s)


# 13. Những lỗi dễ mắc

## Lỗi 1 — Dùng đệ quy cho cây sâu

Viết DFS đệ quy rất tự nhiên, nhưng với đường thẳng dài, Python sẽ báo `RecursionError`:

In [15]:
def subtree_sizes_recursive(n_, edges_):
    adj_ = [[] for _ in range(n_)]
    for u, v in edges_:
        adj_[u].append(v)
        adj_[v].append(u)
    size_ = [1] * n_

    def dfs(u, p):
        for v in adj_[u]:
            if v != p:
                dfs(v, u)
                size_[u] += size_[v]

    dfs(0, -1)
    return size_


try:
    subtree_sizes_recursive(N, chain_edges)
except RecursionError as e:
    print("RecursionError:", str(e)[:60])

RecursionError: maximum recursion depth exceeded


Có thể tăng giới hạn bằng `sys.setrecursionlimit`, nhưng với độ sâu `10^5` vẫn có nguy cơ tràn bộ nhớ stack thật của chương trình (crash). Duyệt bằng stack tự quản lý là cách an toàn.

## Lỗi 2 — Quên phương án "không còn đỉnh đỏ nào"

Nếu lấy `best = min f(u)` mà **không** so với `0`, ví dụ 4 (`"BBB"`) sẽ sai: mọi `f(u)` đều dương (đưa đỉnh xanh vào `S` chỉ tốn tiền), nên code sai buộc phải giữ ít nhất một đỉnh đỏ.

In [16]:
def wrong_no_empty(n_, colors_, edges_):
    parent_, order_ = root_tree(n_, edges_)
    size_ = [1] * n_
    for u in reversed(order_):
        if parent_[u] != -1:
            size_[parent_[u]] += size_[u]
    total_red_ = sum(size_[u] for u in range(n_) if colors_[u] == 'R')
    f_ = [0] * n_
    best_ = float("inf")                       # SAI: thiếu lựa chọn S rỗng (best = 0)
    for u in reversed(order_):
        f_[u] += size_[u] if colors_[u] == 'B' else -size_[u]
        best_ = min(best_, f_[u])
        if parent_[u] != -1:
            f_[parent_[u]] += min(0, f_[u])
    return total_red_ + best_


print("Sai :", wrong_no_empty(3, "BBB", [[0, 1], [1, 2]]))
print("Đúng:", sol.minCost(3, "BBB", [[0, 1], [1, 2]]))

Sai : 1
Đúng: 0


## Lỗi 3 — Cộng `f(c)` của mọi con thay vì `min(0, f(c))`

Như vậy là ép tập `S` phải chứa **toàn bộ** cây con — mất quyền "cắt bỏ nhánh lỗ".

In [17]:
def wrong_take_all_children(n_, colors_, edges_):
    parent_, order_ = root_tree(n_, edges_)
    size_ = [1] * n_
    for u in reversed(order_):
        if parent_[u] != -1:
            size_[parent_[u]] += size_[u]
    total_red_ = sum(size_[u] for u in range(n_) if colors_[u] == 'R')
    f_ = [0] * n_
    best_ = 0
    for u in reversed(order_):
        f_[u] += size_[u] if colors_[u] == 'B' else -size_[u]
        best_ = min(best_, f_[u])
        if parent_[u] != -1:
            f_[parent_[u]] += f_[u]                 # SAI: phải là min(0, f_[u])
    return total_red_ + best_


print("Sai :", wrong_take_all_children(7, "BRBRRBR", edges))
print("Đúng:", sol.minCost(7, "BRBRRBR", edges))

Sai : 1
Đúng: 1


Ở đây code sai vô tình vẫn ra đúng vì tập tốt nhất `{1, 3, 4}` chứa trọn cây con của đỉnh `1`. Một ví dụ khiến nó lộ ra:

In [18]:
# gốc đỏ có hai con: một lá đỏ và một lá xanh
print("Sai :", wrong_take_all_children(3, "RRB", [[0, 1], [0, 2]]))
print("Đúng:", sol.minCost(3, "RRB", [[0, 1], [0, 2]]))

Sai : 1
Đúng: 0


Với `"RRB"` (gốc đỏ, lá `1` đỏ, lá `2` xanh), các đỉnh đỏ đã liên thông → đáp án `0`. Code sai buộc đưa cả lá xanh `2` vào `S` (tốn `1`).

## Lỗi 4 — Coi cạnh là có hướng `edges[i][0] → edges[i][1]`

Đề cho cạnh **vô hướng**; `[u, v]` không có nghĩa `u` là cha của `v`. Test `"cạnh ghi ngược chiều"` ở mục 9 (`[[1, 0], [2, 1]]`) bắt lỗi này. Luôn dựng danh sách kề hai chiều rồi tự tìm cha bằng cách duyệt từ gốc `0`.

## Lỗi 5 — Tính chi phí lật là `1` thay vì kích thước cây con

Đọc lướt đề rất dễ nghĩ mỗi lần lật tốn `1`. Ví dụ 3 sẽ vẫn ra `2` (lá có kích thước 1), nên phải thử thêm test có lật đỉnh trong — như ví dụ xuyên suốt.

## Lỗi 6 — Tính `size` theo sai thứ tự

Duyệt `order` **xuôi** (cha trước con) thì khi cộng `size[u]` vào cha, `size[u]` chưa có đủ các con → kết quả sai. Phải duyệt **ngược**.

# 14. Mẹo nhận diện

1. Đề yêu cầu một **tập đỉnh liên thông** trong **cây**, chi phí/lợi ích **cộng dồn theo từng đỉnh** → nghĩ tới DP **"tập liên thông tốt nhất có đỉnh cao nhất là `u`"**:

   ```text
   f(u) = w[u] + Σ max(0, f(c))     (bài toán tìm tổng lớn nhất)
   f(u) = w[u] + Σ min(0, f(c))     (bài toán tìm tổng nhỏ nhất)
   ```

   Bài toán gốc có tên **maximum-weight connected subtree** (cây con liên thông trọng số lớn nhất).

2. Chi phí phụ thuộc "trạng thái cuối khác trạng thái đầu" → thử **đổi mốc**: chọn một phương án mốc hợp lệ, viết lại chi phí thành "mốc + tổng lời/lỗ của các lựa chọn".
3. `n` lên tới `10^5` trên cây → **luôn duyệt bằng stack**, xử lý theo `reversed(order)` cho các giá trị tính từ con lên cha.

Các bài có cùng "khẩu vị": *Binary Tree Maximum Path Sum* (LeetCode 124 — cũng có `max(0, nhánh con)`), *Maximum Score After Applying Operations on a Tree* (2925), *Count Number of Possible Root Nodes* (2581), và bài 4112 trong cùng bộ notebook này.

# 15. Biến thể

## Biến thể 1 — Bắt buộc còn ít nhất một đỉnh đỏ

Bỏ lựa chọn `S` rỗng: `answer = total_red + min_u f(u)`.

In [19]:
def min_cost_nonempty(n_: int, colors_: str, edges_: List[List[int]]) -> int:
    parent_, order_ = root_tree(n_, edges_)
    size_ = [1] * n_
    for u in reversed(order_):
        if parent_[u] != -1:
            size_[parent_[u]] += size_[u]
    total_red_ = sum(size_[u] for u in range(n_) if colors_[u] == 'R')
    f_ = [0] * n_
    best_ = float("inf")                       # không cho phép S rỗng
    for u in reversed(order_):
        f_[u] += size_[u] if colors_[u] == 'B' else -size_[u]
        best_ = min(best_, f_[u])
        if parent_[u] != -1:
            f_[parent_[u]] += min(0, f_[u])
    return total_red_ + best_


print(min_cost_nonempty(3, "BBB", [[0, 1], [1, 2]]))   # phải lật một lá xanh: 1
print(min_cost_nonempty(3, "RBR", [[0, 1], [1, 2]]))   # giống bài gốc: 1

1
1


(Đây chính là hàm `wrong_no_empty` ở Lỗi 2 — sai với đề gốc nhưng đúng với biến thể này. Đọc đề kỹ quyết định một dòng code!)

## Biến thể 2 — Mỗi lần lật tốn đúng `1` (số lần lật ít nhất)

Chỉ cần thay `size[u]` bằng `1` trong trọng số: `w[u] = +1` nếu xanh, `-1` nếu đỏ.

In [20]:
def min_flips(n_: int, colors_: str, edges_: List[List[int]]) -> int:
    parent_, order_ = root_tree(n_, edges_)
    total_red_ = colors_.count('R')
    f_ = [0] * n_
    best_ = 0
    for u in reversed(order_):
        f_[u] += 1 if colors_[u] == 'B' else -1
        best_ = min(best_, f_[u])
        if parent_[u] != -1:
            f_[parent_[u]] += min(0, f_[u])
    return total_red_ + best_


print(min_flips(4, "BRRR", [[0, 1], [0, 2], [0, 3]]))   # lật gốc thành đỏ: 1 lần
print(min_flips(3, "RBR", [[0, 1], [1, 2]]))            # 1

1
1


Để ý ví dụ 3 (`"BRRR"`): khi mọi lần lật đều tốn `1`, lật **gốc** thành đỏ (1 lần) rẻ hơn lật hai lá (2 lần) — ngược với bài gốc.

---

# 16. Template tư duy

```python
# 1. Dựng cây: danh sách kề hai chiều, duyệt stack từ gốc -> parent, order
# 2. Giá trị từ con lên cha: for u in reversed(order): ...
# 3. DP tập liên thông có top u:
f = [0] * n
best = 0                                   # 0 nếu cho phép tập rỗng
for u in reversed(order):
    f[u] += w[u]                           # f[u] đã chứa Σ min(0, f(con))
    best = min(best, f[u])
    if parent[u] != -1:
        f[parent[u]] += min(0, f[u])       # nhánh lời thì lấy, lỗ thì bỏ
answer = baseline + best
```

# 17. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Cho cây:

```python
n = 6, colors = "RBRRBR", edges = [[0, 1], [1, 2], [1, 3], [0, 4], [4, 5]]
```

1. Vẽ cây với gốc `0`.
2. Tính `size`, `w`, `total_red`.
3. Tính `f` theo thứ tự con trước cha, tìm đáp án.

Kiểm tra bằng cell dưới.

In [21]:
# Kiểm tra Bài 1
trace_tree_dp(6, "RBRRBR", [[0, 1], [1, 2], [1, 3], [0, 4], [4, 5]]);

size = [6, 3, 1, 1, 2, 1], total_red = 9
u = 2 (R): w =  -1, từ các con =   0, f =  -1 -> gửi lên cha 1: -1 (lấy nhánh)
u = 3 (R): w =  -1, từ các con =   0, f =  -1 -> gửi lên cha 1: -1 (lấy nhánh)
u = 1 (B): w =   3, từ các con =  -2, f =   1 -> gửi lên cha 0: 0 (bỏ nhánh)
u = 5 (R): w =  -1, từ các con =   0, f =  -1 -> gửi lên cha 4: -1 (lấy nhánh)
u = 4 (B): w =   2, từ các con =  -1, f =   1 -> gửi lên cha 0: 0 (bỏ nhánh)
u = 0 (R): w =  -6, từ các con =   0, f =  -6 
min f = -6 (top = 0) -> đáp án = 9 + (-6) = 3


## Bài 2 — Dự đoán output

```python
parent_, order_ = root_tree(4, [[2, 0], [0, 1], [3, 1]])
print(parent_)
print(order_)
```

Gợi ý: stack lấy phần tử **cuối** ra trước.

In [22]:
parent_, order_ = root_tree(4, [[2, 0], [0, 1], [3, 1]])
print(parent_)
print(order_)

[-1, 0, 0, 1]
[0, 1, 3, 2]


## Bài 3 — Hoàn thiện phần DP

In [23]:
def exercise_3_min_cost(n: int, colors: str, edges: List[List[int]]) -> int:
    parent, order = root_tree(n, edges)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]
    # TODO 1: tính total_red
    # TODO 2: duyệt reversed(order): f[u] += w[u]; cập nhật best; gửi min(0, f[u]) lên cha
    # TODO 3: return total_red + best (nhớ best bắt đầu từ 0)
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_min_cost(7, "BRBRRBR", [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]]) == 1
# assert exercise_3_min_cost(3, "BBB", [[0, 1], [1, 2]]) == 0

## Bài 4 — Tự viết hàm tính kích thước cây con không đệ quy

Viết `subtree_sizes(n, edges)` trả về list `size`, chạy được với đường thẳng `10^5` đỉnh.

In [24]:
def exercise_4_subtree_sizes(n: int, edges: List[List[int]]) -> List[int]:
    # TODO
    raise NotImplementedError


# assert exercise_4_subtree_sizes(7, [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]]) == [7, 3, 3, 1, 1, 1, 1]
# assert exercise_4_subtree_sizes(10**5, [[i, i + 1] for i in range(10**5 - 1)])[0] == 10**5

## Bài 5 — Cụm đỏ có tổng "giá trị" lớn nhất

Mỗi đỉnh có giá trị `val[u]` (có thể âm). Chọn một tập đỉnh **liên thông khác rỗng** có tổng giá trị **lớn nhất**.

```python
max_connected_sum(5, [3, -5, 4, 2, -1], [[0, 1], [1, 2], [1, 3], [0, 4]])   -> 4
max_connected_sum(3, [-2, -3, -1], [[0, 1], [1, 2]])                        -> -1
```

Gợi ý: đây là "bài gốc" của kỹ thuật, với `max` thay cho `min`.

In [25]:
def exercise_5_max_connected_sum(n: int, val: List[int], edges: List[List[int]]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_max_connected_sum(5, [3, -5, 4, 2, -1], [[0, 1], [1, 2], [1, 3], [0, 4]]) == 4
# assert exercise_5_max_connected_sum(3, [-2, -3, -1], [[0, 1], [1, 2]]) == -1

## Bài 6 — Mini challenge: chỉ ra tập đỏ cuối cùng

Viết hàm trả về **cả chi phí nhỏ nhất và một tập đỉnh đỏ cuối cùng** đạt chi phí đó (dạng list đã sắp xếp).

```python
min_cost_with_set(7, "BRBRRBR", [[0,1],[0,2],[1,3],[1,4],[2,5],[2,6]])  -> (1, [1, 3, 4])
min_cost_with_set(3, "BBB", [[0, 1], [1, 2]])                         -> (0, [])
```

Gợi ý: nhớ đỉnh top đạt `min f`, rồi đi xuống từ top, chỉ đi vào những con `c` có `f(c) < 0`.

In [26]:
def exercise_6_min_cost_with_set(n: int, colors: str, edges: List[List[int]]):
    # TODO
    raise NotImplementedError


# assert exercise_6_min_cost_with_set(7, "BRBRRBR", [[0,1],[0,2],[1,3],[1,4],[2,5],[2,6]]) == (1, [1, 3, 4])
# assert exercise_6_min_cost_with_set(3, "BBB", [[0, 1], [1, 2]]) == (0, [])

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `total_red = Σ size[u]` với `colors[u] == 'R'`.
- Mức 2: `w[u] = size[u]` nếu xanh, `-size[u]` nếu đỏ.
- Mức 3: `f[u] += w[u]; best = min(best, f[u]); if parent[u] != -1: f[parent[u]] += min(0, f[u])`.

**Bài 4**

- Mức 1: dùng `root_tree` (hoặc tự viết lại) để có `parent` và `order`.
- Mức 2: khởi tạo `size = [1] * n`.
- Mức 3: `for u in reversed(order): if parent[u] != -1: size[parent[u]] += size[u]`.

**Bài 5**

- Mức 1: `g(u) = val[u] + Σ max(0, g(c))` là tổng lớn nhất của tập liên thông có top `u`.
- Mức 2: đáp án là `max_u g(u)` (tập phải khác rỗng nên **không** so với `0`).
- Mức 3: tính theo `reversed(order)`, cộng `max(0, g[u])` lên cha.

**Bài 6**

- Mức 1: chạy DP như cũ, nhưng lưu thêm `best_node` (top đạt `min f`, hoặc `None` nếu `S` rỗng tốt nhất).
- Mức 2: cần danh sách con `children[u]` để đi xuống.
- Mức 3: từ `best_node`, dùng stack: lấy `u` vào `S`, đẩy các con `c` có `f[c] < 0`.

# 18. Đáp án tham khảo

Bạn nên tự làm trước khi mở phần này.

## Đáp án Bài 1

```text
          0(R)
         /    \
      1(B)    4(B)
     /    \      \
   2(R)  3(R)    5(R)
```

- `size = [6, 3, 1, 1, 2, 1]`, `w = [-6, 3, -1, -1, 2, -1]`.
- `total_red = size[0] + size[2] + size[3] + size[5] = 6 + 1 + 1 + 1 = 9`.
- `f(2) = -1`, `f(3) = -1`, `f(1) = 3 - 1 - 1 = 1`, `f(5) = -1`, `f(4) = 2 - 1 = 1`, `f(0) = -6 + min(0, 1) + min(0, 1) = -6`.
- `min f = -6` (top `0`, tập `S = {0}`) → đáp án `9 - 6 = 3`: giữ gốc đỏ, lật ba lá đỏ `2, 3, 5` (mỗi lá tốn `1`).

## Đáp án Bài 2

Cạnh: `2—0`, `0—1`, `3—1`. Từ gốc `0`: kề `[2, 1]` → đẩy `2` rồi `1`; lấy `1` ra trước (cuối stack), đẩy `3`; lấy `3`; cuối cùng lấy `2`.

```text
[-1, 0, 0, 1]
[0, 1, 3, 2]
```

In [27]:
# Đáp án Bài 3
def answer_3_min_cost(n: int, colors: str, edges: List[List[int]]) -> int:
    parent, order = root_tree(n, edges)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]
    total_red = sum(size[u] for u in range(n) if colors[u] == 'R')
    f = [0] * n
    best = 0
    for u in reversed(order):
        f[u] += size[u] if colors[u] == 'B' else -size[u]
        best = min(best, f[u])
        if parent[u] != -1:
            f[parent[u]] += min(0, f[u])
    return total_red + best


assert answer_3_min_cost(7, "BRBRRBR", [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]]) == 1
assert answer_3_min_cost(3, "BBB", [[0, 1], [1, 2]]) == 0
assert answer_3_min_cost(6, "RBRRBR", [[0, 1], [1, 2], [1, 3], [0, 4], [4, 5]]) == 3
print("Bài 3: PASS")

Bài 3: PASS


In [28]:
# Đáp án Bài 4
def answer_4_subtree_sizes(n: int, edges: List[List[int]]) -> List[int]:
    parent, order = root_tree(n, edges)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]
    return size


assert answer_4_subtree_sizes(7, [[0, 1], [0, 2], [1, 3], [1, 4], [2, 5], [2, 6]]) == [7, 3, 3, 1, 1, 1, 1]
assert answer_4_subtree_sizes(10**5, [[i, i + 1] for i in range(10**5 - 1)])[0] == 10**5
print("Bài 4: PASS")

Bài 4: PASS


In [29]:
# Đáp án Bài 5
def answer_5_max_connected_sum(n: int, val: List[int], edges: List[List[int]]) -> int:
    parent, order = root_tree(n, edges)
    g = [0] * n
    best = float("-inf")
    for u in reversed(order):
        g[u] += val[u]
        best = max(best, g[u])
        if parent[u] != -1:
            g[parent[u]] += max(0, g[u])
    return best


assert answer_5_max_connected_sum(5, [3, -5, 4, 2, -1], [[0, 1], [1, 2], [1, 3], [0, 4]]) == 4
assert answer_5_max_connected_sum(3, [-2, -3, -1], [[0, 1], [1, 2]]) == -1
print("Bài 5: PASS")

Bài 5: PASS


In [30]:
# Đáp án Bài 6
def answer_6_min_cost_with_set(n: int, colors: str, edges: List[List[int]]):
    parent, order = root_tree(n, edges)
    children = [[] for _ in range(n)]
    for u in range(n):
        if parent[u] != -1:
            children[parent[u]].append(u)
    size = [1] * n
    for u in reversed(order):
        if parent[u] != -1:
            size[parent[u]] += size[u]
    total_red = sum(size[u] for u in range(n) if colors[u] == 'R')
    f = [0] * n
    best, best_node = 0, None
    for u in reversed(order):
        f[u] += size[u] if colors[u] == 'B' else -size[u]
        if f[u] < best:
            best, best_node = f[u], u
        if parent[u] != -1:
            f[parent[u]] += min(0, f[u])

    S = []
    if best_node is not None:
        stack = [best_node]
        while stack:
            u = stack.pop()
            S.append(u)
            stack.extend(c for c in children[u] if f[c] < 0)
    return total_red + best, sorted(S)


assert answer_6_min_cost_with_set(7, "BRBRRBR", [[0,1],[0,2],[1,3],[1,4],[2,5],[2,6]]) == (1, [1, 3, 4])
assert answer_6_min_cost_with_set(3, "BBB", [[0, 1], [1, 2]]) == (0, [])
# kiểm tra: tập trả về phải liên thông và có đúng chi phí đã báo
random.seed(6)
for _ in range(500):
    n_ = random.randint(1, 9)
    edges_ = [[random.randrange(i), i] for i in range(1, n_)]
    colors_ = "".join(random.choice("RB") for _ in range(n_))
    cost, S = answer_6_min_cost_with_set(n_, colors_, edges_)
    adj_ = [[] for _ in range(n_)]
    for u, v in edges_:
        adj_[u].append(v)
        adj_[v].append(u)
    size_ = answer_4_subtree_sizes(n_, edges_)
    real = sum(size_[u] for u in range(n_) if (u in S) != (colors_[u] == 'R'))
    assert is_connected_set(set(S), adj_) and real == cost == min_cost_bruteforce(n_, colors_, edges_)
print("Bài 6: PASS (tập trả về liên thông, đúng chi phí, khớp vét cạn)")

Bài 6: PASS (tập trả về liên thông, đúng chi phí, khớp vét cạn)


# 19. Tổng kết

Bài **Minimum Cost to Make Red Nodes Connected** là một bài tree DP kinh điển "đội lốt":

1. **Trạng thái cuối quyết định chi phí** → bài toán chọn tập đỏ cuối cùng `S`.
2. **Đổi mốc** "lật hết đỏ" → `cost(S) = total_red + Σ_{u∈S} w[u]`, với `w = +size` (xanh) / `-size` (đỏ).
3. **Mỗi tập liên thông có một top duy nhất** → phân loại theo top.
4. **DP:** `f(u) = w[u] + Σ min(0, f(c))`, tính một lượt từ lá lên gốc.

## Công thức nhớ nhanh

```text
answer = total_red + min(0, min_u f(u))
f(u)   = w[u] + Σ_{con c} min(0, f(c))
w[u]   = +size[u] (xanh) | -size[u] (đỏ)
```

## Phiên bản nên nhớ (phần lõi)

```python
f = [0] * n
best = 0
total_red = 0
for u in reversed(order):
    if colors[u] == 'R':
        total_red += size[u]
        f[u] -= size[u]
    else:
        f[u] += size[u]
    best = min(best, f[u])
    if parent[u] != -1:
        f[parent[u]] += min(0, f[u])
return total_red + best
```

## Độ phức tạp

```text
Time:  O(n)
Space: O(n)
```

# 20. Checklist trước khi submit

- [ ] Dựng danh sách kề **hai chiều** (cạnh vô hướng), tự tìm cha bằng cách duyệt từ gốc `0`?
- [ ] Duyệt cây bằng **stack**, không đệ quy (cây sâu `10^5`)?
- [ ] Tính `size` bằng cách duyệt `order` **ngược** (con trước cha)?
- [ ] Chi phí lật là **kích thước cây con**, không phải `1`?
- [ ] Trọng số: xanh `+size`, đỏ `-size`; mốc `total_red` = tổng `size` của các đỉnh đỏ?
- [ ] Gửi lên cha `min(0, f[u])` (bỏ nhánh lỗ), không phải `f[u]`?
- [ ] `best` khởi tạo bằng `0` (cho phép không còn đỉnh đỏ nào)?
- [ ] Đã test: một đỉnh, toàn xanh, toàn đỏ liên thông, cạnh ghi ngược chiều, đường thẳng dài?